##### Cell 1 -> Imports & Regex * extractor function

In [ ]:
# source venv/bin/activate
import re
import pdfplumber
import pandas as pd

START_HDR_RE = re.compile(r"\bOTHER\s+THAN\s+SHOP\s+MATERIALS\b", re.IGNORECASE)
STOP_HDR_RE  = re.compile(r"\bPIECE\s+MARKS\b", re.IGNORECASE)

IDENT_RE = re.compile(r"^(PS[\-A-Z0-9]*|C[A-Z0-9]{6,})$", re.IGNORECASE)
QTY_RE   = re.compile(r"^\d+(\.\d+)?$")

def _cluster_words_into_lines(words, y_tol=3):
    if not words:
        return []
    words = sorted(words, key=lambda w: (w["top"], w["x0"]))
    lines = []
    cur = [words[0]]
    cur_top = words[0]["top"]
    for w in words[1:]:
        if abs(w["top"] - cur_top) <= y_tol:
            cur.append(w)
        else:
            lines.append(sorted(cur, key=lambda x: x["x0"]))
            cur = [w]
            cur_top = w["top"]
    lines.append(sorted(cur, key=lambda x: x["x0"]))
    return lines

def _find_header_x(words):
    targets = {"NO", "NPD", "DESCRIPTION", "IDENT", "QTY"}
    found = {}
    for w in words:
        t = w["text"].strip().upper()
        if t in targets and t not in found:
            found[t] = w["x0"]
    if not targets.issubset(found.keys()):
        return None
    return {k: found[k] for k in ["NO", "NPD", "DESCRIPTION", "IDENT", "QTY"]}

def _compute_boundaries(hx):
    x_no   = hx["NO"]
    x_npd  = hx["NPD"]
    x_desc = hx["DESCRIPTION"]
    x_ident= hx["IDENT"]
    x_qty  = hx["QTY"]

    b_no_npd     = (x_no + x_npd) / 2
    b_npd_desc   = (x_npd + x_desc) / 2
    b_desc_ident = (x_desc + x_ident) / 2
    b_ident_qty  = (x_ident + x_qty) / 2

    return {
        "NO":    (float("-inf"), b_no_npd),
        "NPD":   (b_no_npd, b_npd_desc),
        "DESC":  (b_npd_desc, b_desc_ident),
        "IDENT": (b_desc_ident, b_ident_qty),
        "QTY":   (b_ident_qty, float("inf")),
    }

def extract_everything_below_other_than_shop(pdf_path: str):
    rows = []
    meta = {"start_found": False, "stop_found": False, "rows_parsed": 0}

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            W, H = page.width, page.height

            # Right-side BOM crop (keeps NO column for your files)
            crop_x0 = W * 0.68
            p = page.crop((crop_x0, 0, W, H))

            words = p.extract_words(use_text_flow=False, keep_blank_chars=False)
            if not words:
                continue

            hx = _find_header_x(words)
            if hx is None:
                p2 = page.crop((W * 0.62, 0, W, H))
                words2 = p2.extract_words(use_text_flow=False, keep_blank_chars=False)
                hx = _find_header_x(words2)
                if hx is None:
                    continue
                p = p2
                words = words2

            bounds = _compute_boundaries(hx)
            lines = _cluster_words_into_lines(words, y_tol=3)

            start_y = None
            stop_y = None
            for lw in lines:
                text = " ".join(w["text"] for w in lw)
                if start_y is None and START_HDR_RE.search(text):
                    start_y = min(w["top"] for w in lw)
                elif start_y is not None and STOP_HDR_RE.search(text):
                    stop_y = min(w["top"] for w in lw)
                    break

            if start_y is None:
                continue

            meta["start_found"] = True
            if stop_y is not None:
                meta["stop_found"] = True

            y_min = start_y + 8
            y_max = stop_y if stop_y is not None else H

            win_words = [w for w in words if (w["top"] >= y_min and w["top"] <= y_max)]
            win_lines = _cluster_words_into_lines(win_words, y_tol=3)

            current_section = "OTHER THAN SHOP MATERIALS"
            current = None

            def flush():
                nonlocal current
                if current:
                    rows.append(current)
                    meta["rows_parsed"] += 1
                    current = None

            def bucketize(line_words):
                buckets = {"NO": [], "NPD": [], "DESC": [], "IDENT": [], "QTY": []}
                for w in line_words:
                    x = w["x0"]
                    t = w["text"].strip()
                    for k, (lo, hi) in bounds.items():
                        if lo < x <= hi:
                            buckets[k].append(t)
                            break
                return buckets

            def spill_text_from_ident(b):
                # keep only tokens that are NOT real idents (so "PL-0503" can still come through)
                return " ".join(t for t in b["IDENT"] if not IDENT_RE.match(t)).strip()

            def spill_text_from_qty(b):
                # keep anything in QTY bucket that is NOT a numeric qty (Valve tags often land here)
                return " ".join(t for t in b["QTY"] if not QTY_RE.match(t)).strip()

            for lw in win_lines:
                line_text = " ".join(w["text"] for w in lw).strip()
                if not line_text:
                    continue

                if re.fullmatch(r"PIPE\s+SUPPORTS", line_text, flags=re.IGNORECASE):
                    flush()
                    current_section = "PIPE SUPPORTS"
                    continue

                if re.search(r"\bNO\b.*\bNPD\b.*\bDESCRIPTION\b.*\bIDENT\b.*\bQTY\b", line_text, re.IGNORECASE):
                    continue

                b = bucketize(lw)
                no_val = next((t for t in b["NO"] if t.isdigit()), None)

                if no_val:
                    flush()

                    npd_val = " ".join(b["NPD"]).strip()

                    ident_val = next((t for t in b["IDENT"] if IDENT_RE.match(t)), "")

                    qty_val = ""
                    for t in reversed(b["QTY"]):
                        if QTY_RE.match(t):
                            qty_val = t
                            break

                    desc_val = " ".join(b["DESC"]).strip()

                    ident_spill = spill_text_from_ident(b)
                    if ident_spill:
                        desc_val = (desc_val + " " + ident_spill).strip()

                    qty_spill = spill_text_from_qty(b)
                    if qty_spill:
                        desc_val = (desc_val + " " + qty_spill).strip()

                    current = {
                        "section": current_section,
                        "no": no_val,
                        "npd": npd_val,
                        "description": desc_val,
                        "ident": ident_val,
                        "qty": qty_val,
                    }
                else:
                    # continuation line -> append desc + ident spill + qty spill
                    if current:
                        cont_desc = " ".join(b["DESC"]).strip()
                        cont_ident = spill_text_from_ident(b)
                        cont_qty = spill_text_from_qty(b)

                        cont = " ".join(x for x in [cont_desc, cont_ident, cont_qty] if x).strip()
                        if cont:
                            current["description"] = (current["description"] + " " + cont).strip()

            flush()

            if meta["stop_found"]:
                break

    df = pd.DataFrame(rows)
    return df, meta


#### Cell 2 -> Loop through all pdfs, build all rows, concat into df

In [ ]:
from pathlib import Path
import pandas as pd

# ---- CONFIG ----
PDF_DIR = Path("6820 Copy")
LIMIT = None  # set to None for all PDFs

pdfs = [pdf_path for pdf_path in sorted(PDF_DIR.glob("*.pdf")) if "OLD REV_S" not in pdf_path.parts]
if LIMIT is not None:
    pdfs = pdfs[:LIMIT]

out, fails = [], []

for p in pdfs:
    try:
        df, meta = extract_everything_below_other_than_shop(str(p))

        if df.empty:
            reason = "Header not found" if not meta.get("start_found") else "Header found but no rows parsed"
            fails.append({"source_file": p.name, "reason": reason, **meta})
            continue

        # Add file metadata columns
        df.insert(0, "source_file", p.name)

        # ISO like 6820-R2-67001-01 (you can adjust pattern if needed)
        df["iso"] = df["source_file"].str.extract(r"(6820-R2-\d{5}-\d{2})", expand=False)

        # Area (left side of ISO) -> 6820
        df["area"] = df["iso"].str.extract(r"^(\d{4})", expand=False)

        out.append(df)

    except Exception as e:
        fails.append({
            "source_file": p.name,
            "reason": repr(e),
            "start_found": None,
            "stop_found": None,
            "rows_parsed": None
        })

all_df = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
fail_df = pd.DataFrame(fails)

print("PDFs tested:", len(pdfs))
print("Rows extracted:", len(all_df))
print("Failures:", len(fail_df))

display(all_df.head(30))
if not fail_df.empty:
    display(fail_df.sort_values(["reason", "source_file"]).reset_index(drop=True))
else:
    print("✅ No failures in this batch.")


#####  Quarentine

In [ ]:
# --- NEXT CELL: inspect failures + build a material summary ---

# 1) Look at failures (if any)
if not fail_df.empty:
    display(fail_df.sort_values(["reason", "source_file"]).reset_index(drop=True))
else:
    print("✅ No failures in this batch.")

# 2) Clean types + build a simple planner summary
if not all_df.empty:
    # qty as numeric
    all_df["qty"] = pd.to_numeric(all_df["qty"], errors="coerce").fillna(0)

    # quick summary: total qty by ident + npd (and section if you want)
    material_summary = (
        all_df.groupby(["ident", "npd"], as_index=False)
              .agg(total_qty=("qty", "sum"),
                   rows=("qty", "size"),
                   isos=("iso", "nunique"))
              .sort_values(["ident", "npd"])
    )

    display(material_summary.head(50))
else:
    print("⚠️ all_df is empty — nothing to summarize.")


In [ ]:
import re
import pandas as pd

if all_df.empty:
    print("⚠️ all_df is empty")
else:
    dfq = all_df.copy()

    # strong typing
    dfq["qty"] = pd.to_numeric(dfq["qty"], errors="coerce").fillna(0)
    dfq["no"] = pd.to_numeric(dfq["no"], errors="coerce")
    dfq["ident"] = dfq["ident"].astype(str).str.strip()
    dfq["npd"] = dfq["npd"].astype(str).str.strip()
    dfq["description"] = dfq["description"].astype(str).str.strip()

    ONLY_DIGITS = re.compile(r"^\d+$")
    NPD_OK = re.compile(r'^\d+(?:/\d+)?(?:\.\d+)?(?:"?)$')  # 8, 3/4, 1/2, 10", 2.5 etc (quotes optional)

    def flag_reason(r):
        ident = r["ident"]
        npd = r["npd"]
        no = r["no"]
        qty = r["qty"]

        # obvious bad parses
        if pd.isna(no) or no <= 0 or no > 500:
            return "bad_no"
        if qty <= 0:
            return "bad_qty"
        if ident == "" or len(ident) < 2:
            return "bad_ident"
        if ONLY_DIGITS.match(ident):
            return "ident_digits_only"
        if not NPD_OK.match(npd):
            return "npd_suspicious"

        # if it looks like a pure coordinate/geometry spill in description (optional)
        # keep it REVIEW, not junk
        if "EL +" in r["description"] or "T.O.S" in r["description"]:
            return "desc_has_coords"

        return ""

    dfq["flag"] = dfq.apply(flag_reason, axis=1)

    good_df = dfq[dfq["flag"] == ""].copy()
    review_df = dfq[dfq["flag"] != ""].copy()

    print("Total rows:", len(dfq))
    print("Good rows:", len(good_df))
    print("Review rows:", len(review_df))

    display(review_df[["source_file","iso","section","no","npd","ident","qty","flag","description"]].head(30))


##### Cell 4 -> Planner outputs + Export

In [ ]:
import pandas as pd

if good_df.empty:
    print("⚠️ good_df is empty")
else:
    # 1) Material Takeoff (global)
    material_takeoff = (
        good_df
        .groupby(["ident","npd"], as_index=False)
        .agg(
            total_qty=("qty","sum"),
            iso_count=("iso","nunique"),
            rows=("ident","count")
        )
        .sort_values(["ident","npd"])
    )

    # 2) Material by ISO
    material_by_iso = (
        good_df
        .groupby(["iso","ident","npd"], as_index=False)
        .agg(total_qty=("qty","sum"))
        .sort_values(["iso","ident","npd"])
    )

    display(material_takeoff.head(30))
    display(material_by_iso.head(30))

    # 3) Export
    OUTPUT_XLSX = "6820_BOM_Output.xlsx"
    with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
        all_df.to_excel(writer, sheet_name="RAW_ALL", index=False)
        good_df.to_excel(writer, sheet_name="GOOD", index=False)
        review_df.to_excel(writer, sheet_name="REVIEW", index=False)
        material_takeoff.to_excel(writer, sheet_name="TAKEOFF", index=False)
        material_by_iso.to_excel(writer, sheet_name="BY_ISO", index=False)

    good_df.to_csv("good_df.csv", index=False)
    review_df.to_csv("review_df.csv", index=False)
    material_takeoff.to_csv("material_takeoff.csv", index=False)
    material_by_iso.to_csv("material_by_iso.csv", index=False)

    print("✅ Exported:", OUTPUT_XLSX)


##### Final export + audit

In [ ]:
import pandas as pd
from pathlib import Path

# If these dataframes already exist in memory, you can skip reading from CSV.
# Otherwise, load from the CSV outputs you generated:
good_df = pd.read_csv("good_df.csv")
review_df = pd.read_csv("review_df.csv")
material_by_iso = pd.read_csv("material_by_iso.csv")
material_takeoff = pd.read_csv("material_takeoff.csv")

# Quick audit stats
print("GOOD rows:", len(good_df))
print("REVIEW rows:", len(review_df))
if len(review_df):
    print("\nReview flags breakdown:")
    print(review_df["flag"].value_counts(dropna=False))

# Export pack (Excel + CSV folder)
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

good_df.to_csv(OUT_DIR / "GOOD.csv", index=False)
review_df.to_csv(OUT_DIR / "REVIEW.csv", index=False)
material_by_iso.to_csv(OUT_DIR / "BY_ISO.csv", index=False)
material_takeoff.to_csv(OUT_DIR / "TAKEOFF.csv", index=False)

# Excel (use openpyxl to avoid xlsxwriter dependency)
OUTPUT_XLSX = OUT_DIR / "6820_BOM_Output.xlsx"
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    good_df.to_excel(writer, sheet_name="GOOD", index=False)
    review_df.to_excel(writer, sheet_name="REVIEW", index=False)
    material_by_iso.to_excel(writer, sheet_name="BY_ISO", index=False)
    material_takeoff.to_excel(writer, sheet_name="TAKEOFF", index=False)

print("✅ Wrote exports to:", OUT_DIR.resolve())
print("✅ Excel:", OUTPUT_XLSX.resolve())


## Instrument extractor


In [ ]:
from pathlib import Path
import pandas as pd

from scripts.extract_instruments import resolve_pdf_dir, run

summary = run(resolve_pdf_dir(None), Path("outputs"))
print(pd.Series(summary).to_string())

instrument_df = pd.read_csv("outputs/instruments_clean.csv")
instrument_raw_df = pd.read_csv("outputs/instruments_raw.csv")
low_confidence_review_df = pd.read_csv("outputs/low_confidence_review.csv")

display(instrument_df.head(30))
